In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

model = init_chat_model("gemini-3.6-flash", model_provider="google_genai")

### Prompt Templates in LangChain

Prompt Templates are reusable recipes for generating prompts with dynamic input parameters.

Instead of manually formatting strings with f-strings, Prompt Templates handle variable injection, type validation, and conversion into structured message lists (List[BaseMessage]) for chat models.

### Prompt Template Reference Matrix

| Prompt Type | Popularity | Common Use Case |
| :--- | :--- | :--- |
| ChatPromptTemplate | High | Primary template for almost every LangChain chat app |
| MessagesPlaceholder | High | Dynamically injecting chat history and tool scratchpads |
| RAG Prompt | High | Context-augmented generation (Q&A over documents) |
| Structured Output Prompt | High | Extracting structured Pydantic data from LLMs |
| Agent Prompt | High | Tool-calling agents with reasoning & scratchpads |
| FewShotChatMessagePromptTemplate | High | In-context learning via input/output example pairs |
| Partial Prompt (.partial()) | High | Pre-filling static variables or dynamic timestamps |
| Multimodal Prompt | High | Parameterizing text and image URLs for vision models |
| PromptTemplate | High | Legacy string completion models or simple text prompts |
| Pipeline / Modular Composition (+) | High | Combining modular prompt templates together |

### 1. ChatPromptTemplate

ChatPromptTemplate creates structured message lists for chat models using tuple pairs ("role", "template").

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {role}. Explain concepts clearly."),
    ("human", "Explain {concept} concisely.")
])

messages = chat_template.invoke({
    "role": "Python Developer",
    "concept": "list comprehensions"
})

response = model.invoke(messages)
print("Model Response:")
print(response.content)

### 2. MessagesPlaceholder (Chat History & Memory)

MessagesPlaceholder injects dynamic message history lists (List[BaseMessage]) into prompt templates at runtime.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful support bot."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

history = [
    HumanMessage(content="My name is Alex."),
    AIMessage(content="Hello Alex! How can I help you?")
]

messages = chat_template.invoke({
    "chat_history": history,
    "question": "What is my name?"
})

response = model.invoke(messages)
print("Model Response:")
print(response.content)

### 3. RAG Prompt (Retrieval-Augmented Generation)

RAG prompts combine retrieved document context {context} with the user query {question} to anchor the LLM's answer in factual data.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the user's question using ONLY the provided context below. If unknown, say 'I don't know'.\n\nContext:\n{context}"),
    ("human", "{question}")
])

retrieved_context = "LangChain was created by Harrison Chase in October 2022. It is an open-source framework for building applications powered by language models."

messages = rag_prompt.invoke({
    "context": retrieved_context,
    "question": "Who created LangChain and when?"
})

response = model.invoke(messages)
print("RAG Model Response:")
print(response.content)

### 4. Structured Output Prompt

Combines prompt templates with .with_structured_output(PydanticSchema) to extract structured JSON objects directly from LLMs.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

class PersonDetails(BaseModel):
    name: str = Field(description="Name of the person")
    profession: str = Field(description="Profession or role of the person")
    skills: list[str] = Field(description="List of mentioned technical skills")

# Bind Pydantic schema to model
structured_model = model.with_structured_output(PersonDetails)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract person details accurately from the text."),
    ("human", "{input_text}")
])

chain = prompt | structured_model
result = chain.invoke({
    "input_text": "Guido van Rossum is a Dutch programmer who created Python and excels in C, C++, and language architecture."
})

print(f"Parsed Pydantic Object: {type(result)}")
print(f"Name: {result.name}")
print(f"Profession: {result.profession}")
print(f"Skills: {result.skills}")

### 5. Agent Prompt (Tool-Calling Agent Prompt)

Agent prompts include system instructions, dynamic chat history, and an agent_scratchpad placeholder where previous tool execution steps are appended during reasoning loops.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an autonomous AI Agent equipped with tools. Reason step-by-step."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

formatted = agent_prompt.invoke({
    "chat_history": [],
    "input": "Calculate weather in Tokyo",
    "agent_scratchpad": []
})

print("Agent Prompt Messages Stack:")
for m in formatted.to_messages():
    print(f"[{m.type.upper()}]: {m.content}")

### 6. FewShotChatMessagePromptTemplate

FewShotChatMessagePromptTemplate incorporates example input/output pairs into the prompt to guide LLM behavior and format via in-context learning.

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate, ChatPromptTemplate

examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"}
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "Output the antonym of the provided word."),
    few_shot_prompt,
    ("human", "{input}")
])

messages = final_prompt.invoke({"input": "fast"})
response = model.invoke(messages)
print("Model Response:")
print(response.content)

### 7. Partial Prompt Templates (.partial())

Partial formatting allows pre-filling specific variables early (e.g. static roles or dynamic functions like timestamps) before main invocation.

In [ ]:
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate

date_prompt = ChatPromptTemplate.from_messages([
    ("system", "Today's date is {current_date}."),
    ("human", "{user_request}")
]).partial(current_date=lambda: datetime.now().strftime("%Y-%m-%d"))

formatted = date_prompt.invoke({"user_request": "What is today's date?"})
print("System Content:", formatted.to_messages()[0].content)

### 8. Multimodal Prompt Templates

Multimodal prompt templates parameterize text queries and image URLs dynamically for vision models.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

multimodal_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert visual assistant."),
    (
        "human",
        [
            {"type": "text", "text": "{question}"},
            {
                "type": "image_url",
                "image_url": {"url": "{image_url}"}
            }
        ]
    )
])

messages = multimodal_prompt.invoke({
    "question": "Describe the dog in this image.",
    "image_url": "https://cdn.pixabay.com/photo/2015/12/30/21/34/labrador-1114810_1280.jpg"
})

response = model.invoke(messages)
print("Model Response:")
print(response.content)

### 9. PromptTemplate (Text Template)

PromptTemplate is used for generating raw text completion strings.

In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("Summarize key points of {topic} in 3 bullets.")
formatted = template.format(topic="FastAPI")
print(formatted)

### 10. Pipeline & Modular Composition (+ Operator)

You can combine multiple ChatPromptTemplate instances together using the + operator to build modular prompts.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_part = ChatPromptTemplate.from_messages([("system", "You are a code reviewer.")])
human_part = ChatPromptTemplate.from_messages([("human", "Review: {code}")])

full_prompt = system_part + human_part
res = full_prompt.invoke({"code": "def add(a, b): return a + b"})
for m in res.to_messages():
    print(f"[{m.type.upper()}]: {m.content}")